In [1]:
import sys
sys.path.append('../../')
import os
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor
from rover_simulator.navigation.rl_env import RoverGymEnv
from rl.callbacks import EpisodeInfoCallback, RolloutCheckpointCallback
from stable_baselines3.common.callbacks import CheckpointCallback, CallbackList

In [2]:
def make_env(map_file=None):
    def _init():
        return RoverGymEnv(map_file=map_file)
    return _init

In [3]:
timesteps = 1000000
model_dir = "../models"
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, "ppo_rover")

In [ ]:
env = DummyVecEnv([make_env()])
env = VecMonitor(env)   # Wrap with VecMonitor to record episode rewards/lengths for TensorBoard

model = PPO("MlpPolicy", env, verbose=1, tensorboard_log="./rl_logs")

callbacks = []
# episode info -> tensorboard
# periodic checkpoint every N timesteps (default 50k)
# create checkpoint callback saving into model_dir
checkpoint_cb = CheckpointCallback(save_freq=50000, save_path=model_dir, name_prefix="ppo_rover")
# rollout checkpoint (save after every rollout)
rollout_cb = RolloutCheckpointCallback(save_dir=model_dir, name_prefix="ppo_rover")

callbacks.append(EpisodeInfoCallback())
callbacks.append(checkpoint_cb)
callbacks.append(rollout_cb)
cb_list = CallbackList(callbacks)

Using cuda device


/home/motthi/.local/lib/python3.10/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [ ]:
model.learn(total_timesteps=timesteps, callback=cb_list, progress_bar=True)

Logging to ./rl_logs/PPO_3


Output()

-------------------------------------------
| custom/                      |          |
|    cumulative_collision_rate | 0.167    |
|    cumulative_success_rate   | 0.0833   |
|    rollout_collision_rate    | 0.167    |
|    rollout_success_rate      | 0.0833   |
| rollout/                     |          |
|    ep_len_mean               | 157      |
|    ep_rew_mean               | -31.4    |
| time/                        |          |
|    fps                       | 98       |
|    iterations                | 1        |
|    time_elapsed              | 20       |
|    total_timesteps           | 2048     |
-------------------------------------------


----------------------------------------------
| custom/                      |             |
|    cumulative_collision_rate | 0.226       |
|    cumulative_success_rate   | 0.226       |
|    rollout_collision_rate    | 0.263       |
|    rollout_success_rate      | 0.316       |
| rollout/                     |             |
|    ep_len_mean               | 131         |
|    ep_rew_mean               | -17.2       |
| time/                        |             |
|    fps                       | 97          |
|    iterations                | 2           |
|    time_elapsed              | 41          |
|    total_timesteps           | 4096        |
| train/                       |             |
|    approx_kl                 | 0.007810464 |
|    clip_fraction             | 0.11        |
|    clip_range                | 0.2         |
|    entropy_loss              | -2.83       |
|    explained_variance        | -0.00279    |
|    learning_rate             | 0.0003      |
|    loss    

KeyboardInterrupt: 

In [ ]:
model.save(model_path)